# 01 — Simulate small data

Runs `python -m ssm_phylo.data.simulation --smoke` into LOCAL scratch, consolidates + splits, copies the parquet to `$DATA_DIR` atomically, then prints data stats and a patristic-distance histogram. Re-run safe: skipped when `train.parquet` exists unless **overwrite** is checked.

In [ ]:
import os, subprocess, sys, tempfile, pathlib
REPO_DIR = os.path.abspath(os.getcwd())
sys.path.insert(0, REPO_DIR)
DRIVE_SKIP = os.environ.get("COLAB_DRIVE_SKIP", "0") == "1"
if not os.environ.get("COLAB_DRIVE"):
    COLAB_DRIVE = ("/content/drive/MyDrive/ssm-phylo" if not DRIVE_SKIP
                   else tempfile.mkdtemp(prefix="ssm_drive_"))
    os.environ.update(
        COLAB_DRIVE=COLAB_DRIVE,
        DATA_DIR=f"{COLAB_DRIVE}/data",
        LOCAL_CKPT_DIR="/content/ckpts" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_ckpts",
        LOCAL_DATA_DIR="/content/data" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_data",
        CKPT_DIR=f"{COLAB_DRIVE}/checkpoints",
        RESULTS_DIR=f"{COLAB_DRIVE}/results",
    )
    for d in [os.environ["LOCAL_DATA_DIR"], os.environ["LOCAL_CKPT_DIR"],
              os.environ["DATA_DIR"], os.environ["CKPT_DIR"], os.environ["RESULTS_DIR"]]:
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", os.environ["DATA_DIR"])
print("LOCAL_DATA_DIR:", os.environ["LOCAL_DATA_DIR"])


In [ ]:
#@title Simulation controls
overwrite = False  #@param {type:"boolean"}
train_parquet = os.path.join(os.environ["DATA_DIR"], "train.parquet")
if os.path.exists(train_parquet) and not overwrite:
    print("train.parquet exists — skipping simulation (check 'overwrite' to redo)")
else:
    proc = subprocess.run(
        [sys.executable, "-m", "ssm_phylo.data.simulation", "--smoke"],
        env=os.environ, text=True, cwd=REPO_DIR,
    )
    print("simulation exit:", proc.returncode)

In [ ]:
#@title Data stats
import pyarrow.parquet as pq
import numpy as np

for split in ("train", "val", "test"):
    p = os.path.join(os.environ["DATA_DIR"], f"{split}.parquet")
    if os.path.exists(p):
        t = pq.read_table(p)
        tips = t["n_tips"].to_numpy()
        lens = np.array([len(s) for row in t["seqs"].to_pylist()[:200] for s in row])
        print(f"{split}: {len(t)} rows | n_tips {int(tips.min())}-{int(tips.max())} "
              f"| median seq len {int(np.median(lens))}")

In [ ]:
#@title Patristic-distance histogram (sample of the train split)
import matplotlib.pyplot as plt
import dendropy

t = pq.read_table(os.path.join(os.environ["DATA_DIR"], "train.parquet"),
                  columns=["tree_newick"])
dists = []
for nwk in t["tree_newick"].to_pylist()[:50]:
    tree = dendropy.Tree.get(data=nwk, schema="newick")
    ndm = tree.node_distance_matrix()
    leaves = tree.leaf_nodes()
    for i in range(len(leaves)):
        for j in range(i + 1, len(leaves)):
            dists.append(float(ndm(leaves[i], leaves[j])))
plt.figure(figsize=(6, 3))
plt.hist(dists, bins=40)
plt.xlabel("patristic distance (subs/site)")
plt.ylabel("pairs")
plt.title("True pairwise distances (train sample)")
plt.show()
print("n pairs:", len(dists))